# Romanized Telugu → Qwen3-4B-Instruct-2507

Colab notebook for reviewing anonymous conversations, mapping speakers, converting to Qwen chat format, and optionally starting QLoRA. Do not train until you verify that assistant messages are yours.

In [ ]:
# Install dependencies
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub


In [ ]:
from google.colab import drive
# Optional: mount Drive if your data is stored there
drive.mount('/content/drive')


## Get the project

Option A: connect Colab to GitHub using **File → Save a copy in GitHub**.

Option B: clone a private repository (replace the URL):

In [ ]:
# Clone public code; private data stays in Drive.
!rm -rf /content/romanized-telugu-qwen-colab
!git clone -q https://github.com/TharunChougoni/romanized-telugu-qwen-colab.git /content/romanized-telugu-qwen-colab
CODE = '/content/romanized-telugu-qwen-colab'


In [ ]:
# Private data lives in Drive. Upload cleaned_conversations.jsonl here first.
PROJECT = '/content/drive/MyDrive/romanized_telugu_dataset_cleaned'
INPUT = f'{PROJECT}/cleaned_conversations.jsonl'
MAPPING = f'{PROJECT}/speaker_mapping.json'
import os
print(INPUT)
print('Drive files:', os.listdir(PROJECT))


## Review anonymous speakers

The cleaner anonymizes speakers separately per conversation/archive. Do not assume speaker_0 is you globally. Inspect a small sample and create a per-conversation mapping if necessary.

In [ ]:
import json, itertools
with open(INPUT, encoding='utf-8') as f:
    samples = list(itertools.islice(f, 5))
for line in samples:
    row=json.loads(line)
    print('CONVERSATION:', row.get('conversation_id'))
    for m in row['messages']:
        print(m['role'], ':', m['content'][:250])
    print('-'*60)


Create `speaker_mapping.json` with the conversation ID and the anonymous speaker that represents you. Example: `{'abc123...': 'speaker_1'}`. Keep this file private.

In [ ]:
# Only run after reviewing samples; this file remains private in Drive.
# Example: mapping = {'conversation_id_here': 'speaker_1'}
mapping = {}
MAPPING = f'{PROJECT}/speaker_mapping.json'
with open(MAPPING, 'w') as f: json.dump(mapping, f, indent=2)
print('Wrote', MAPPING)


## Convert to Qwen chat format

The converter imports `Qwen/Qwen3-4B-Instruct-2507` from Hugging Face and validates its official chat template. It disables thinking mode for natural conversation training.

## Build LoRA-ready prompt/completion examples

This creates one example per assistant turn with prior turns as context. It preserves the edgy/unhinged wording; this stage does not filter profanity, insults, sexual language, sarcasm, or taboo style. It does split train/validation by conversation to reduce leakage.

In [ ]:
!python {CODE}/src/build_lora_dataset.py --input {INPUT} --output {PROJECT}/qwen_lora_sft --all-speakers --max-context-turns 8

In [ ]:
from pathlib import Path
import json
for name in ['train.jsonl','validation.jsonl']:
    p=Path(PROJECT)/'qwen_lora_sft'/name
    print(name, sum(1 for _ in p.open()))
    lines = p.read_text(encoding='utf-8').splitlines()
    print(lines[0][:1000] if lines else '(empty file)')


In [ ]:
# Inspect the first LoRA-ready example.
from pathlib import Path
for name in ['train.jsonl','validation.jsonl']:
    p=Path(PROJECT)/'qwen_lora_sft'/name
    print(name, sum(1 for _ in p.open()))
    lines = p.read_text(encoding='utf-8').splitlines()
    print(lines[0][:1000] if lines else '(empty file)')


## 1k QLoRA smoke test (T4)

This is the first real training run: 1,000 train examples, 200 validation examples, four context turns, 1,024-token sequences, one epoch. It writes checkpoints to Colab local disk for speed, then copies the final adapter to Drive.

In [ ]:
# Rebuild a smaller-context dataset for the 1k T4 smoke test.
!python {CODE}/src/build_lora_dataset.py --input {INPUT} --output /content/qwen_lora_sft_1k --all-speakers --max-context-turns 4 --max-train-examples 1000 --max-validation-examples 200


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
RUN_DIR = '/content/qwen3-romanized-telugu-1k-lora'
DRIVE_ADAPTER_DIR = f'{PROJECT}/qwen3-romanized-telugu-1k-lora'

files = {'train': '/content/qwen_lora_sft_1k/train.jsonl', 'validation': '/content/qwen_lora_sft_1k/validation.jsonl'}
ds = load_dataset('json', data_files=files)
# Seeded builder order is randomized: first 1k/200 is a representative smoke-test subset.
ds['train'] = ds['train'].select(range(min(1000, len(ds['train']))))
ds['validation'] = ds['validation'].select(range(min(200, len(ds['validation']))))
print(ds)

tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM')
# T4 does not support BF16 training. Load explicitly in FP16, then make LoRA weights FP32.
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, torch_dtype=torch.float16, device_map={'': 0})
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora)
for p in model.parameters():
    if p.requires_grad: p.data = p.data.float()
model.print_trainable_parameters()
args = SFTConfig(
    output_dir=RUN_DIR,
    num_train_epochs=1,
    max_length=1024,
    packing=True,
    assistant_only_loss=True,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    # Do not enable AMP GradScaler on T4: Qwen/PEFT exposes BF16 adapter grads.
    # 4-bit linear layers still compute in FP16 via `bnb_4bit_compute_dtype`.
    fp16=False,
    bf16=False,
    eval_strategy='steps',
    eval_steps=25,
    save_strategy='steps',
    save_steps=25,
    save_total_limit=1,
    logging_steps=5,
    disable_tqdm=False,
    report_to='none',
)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['validation'], processing_class=tokenizer)
trainer.train()
trainer.save_model(RUN_DIR)
tokenizer.save_pretrained(RUN_DIR)
print('Saved locally:', RUN_DIR)


In [ ]:
# Copy only the final LoRA adapter/tokenizer to Drive after training completes.
!rm -rf {DRIVE_ADAPTER_DIR}
!cp -r {RUN_DIR} {DRIVE_ADAPTER_DIR}
!du -sh {DRIVE_ADAPTER_DIR}
print('Adapter saved to:', DRIVE_ADAPTER_DIR)


## Privacy and GitHub checklist

- Keep the repository private.
- Never commit raw WhatsApp ZIPs, contact cards, phone numbers, or secrets.
- Treat even cleaned chat text as private.
- Review samples for residual personal details before training.
- Do not upload the final adapter publicly without consent.